# LLM Experiments

This is the most unique part of the project. Instead of training a model on the data, I'm using an already-trained LLM (Llama 3 70B via Groq API) and asking it to predict PTSD risk from patient descriptions.

The core idea: take the same patient row that XGBoost sees as numbers, convert it into text, and ask the LLM to predict. This is called "tabular-to-text prompting" — LLMs are not designed for this kind of structured clinical data.

I'm trying 4 different ways to describe each patient (serialization formats) and 2 prompting styles (zero-shot and few-shot) = 8 experiments total.

In [35]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install groq -q

import pandas as pd
import numpy as np
import re
import time
from groq import Groq

base = '/content/drive/MyDrive/PTSD_Project'
processed = base + '/data/processed'
results = base + '/results'
models_path = base + '/models'

print("done")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
done


## Loading data

I need two versions of the test data:
- The encoded version (35 columns) — just to get the true labels
- The original unencoded version — to build readable text prompts (age, gender etc. as actual values)

Also loading the SHAP top features from 04_baseline_models — these are used in Format 4 (the SHAP-guided prompt).

In [36]:
# Load data
# encoded test data - for labels only
test_encoded = pd.read_csv(processed + '/test.csv')
y_test = test_encoded['label']

# original unencoded data - for building prompts
merged = pd.read_csv(processed + '/merged_clean.csv')

# top features from SHAP
top_features = pd.read_csv(results + '/top_features_with_values.csv')

print("test_encoded shape:", test_encoded.shape)
print("merged shape:", merged.shape)
print("top_features shape:", top_features.shape)
print("\nlabel distribution in test:")
print(y_test.value_counts())

test_encoded shape: (1013, 35)
merged shape: (5065, 24)
top_features shape: (1013, 3)

label distribution in test:
label
0    921
1     92
Name: count, dtype: int64


In [37]:
# Get original test rows for prompt building
# test_encoded and merged don't share an index directly
# we need to align them using the label column

# reset index on test_encoded to get position
test_encoded = test_encoded.reset_index(drop=True)

# the test set was created from merged_clean with a fixed random state
# we recreate the same split to get the original rows
from sklearn.model_selection import train_test_split

X = merged.drop(columns=['label', 'PHQ_total', 'SEQN'])
y = merged['label']

_, X_test_orig, _, _ = train_test_split(
    merged, y, test_size=0.2, random_state=42, stratify=y
)

X_test_orig = X_test_orig.reset_index(drop=True)

print("original test rows shape:", X_test_orig.shape)
print("\nsample:")
print(X_test_orig[['RIDAGEYR', 'RIAGENDR', 'SLD012', 'DPQ010', 'label']].head())

original test rows shape: (1013, 24)

sample:
   RIDAGEYR  RIAGENDR  SLD012  DPQ010  label
0      42.0       2.0     8.5       0      0
1      74.0       2.0     6.0       0      0
2      21.0       2.0    11.0       1      0
3      45.0       1.0     7.5       1      0
4      56.0       1.0     7.0       1      0


## Setting up Groq API

Groq gives free access to Llama 3 70B — a large language model with strong general knowledge. I'm using this instead of GPT-4 or Claude to keep the project completely free.

Testing the connection with a simple prompt before running experiments.

In [39]:
# Groq API setup
GROQ_API_KEY = "GROQ_API_KEY" #please add you api key

client = Groq(api_key=GROQ_API_KEY)

# test connection
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Reply with just the number 1"}],
    temperature=0
)

print("API working! Response:", response.choices[0].message.content)

API working! Response: 1


## Sampling 200 patients

Running all 1013 test patients through 8 experiments would mean ~8000 API calls which takes too long and could hit rate limits. Instead I'm using a random sample of 200 patients — still enough to get meaningful results.

The call_groq function handles the API call and returns the raw response. max_tokens=10 forces the LLM to give a very short answer.

In [40]:
# Sample 200 patients and build call function
np.random.seed(42)
sample_idx = np.random.choice(len(X_test_orig), size=200, replace=False)

X_sample = X_test_orig.iloc[sample_idx].reset_index(drop=True)
y_sample = y_test.iloc[sample_idx].reset_index(drop=True)
top_features_sample = top_features.iloc[sample_idx].reset_index(drop=True)

print("sample shape:", X_sample.shape)
print("label distribution in sample:")
print(y_sample.value_counts())

def call_groq(prompt):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=10
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return "error"

sample shape: (200, 24)
label distribution in sample:
label
0    185
1     15
Name: count, dtype: int64


## The 4 serialization formats

This is the key design decision — how do I convert a patient row into text? I tried 4 different styles:

- Format 1 (Basic): simple key-value pairs
- Format 2 (Sentence): natural language description  
- Format 3 (Clinical): compact clinical notation
- Format 4 (SHAP-guided): uses XGBoost SHAP values to highlight the most important features for each specific patient — this is the unique addition

Each format asks the LLM to output only 0 or 1.

In [41]:

def format1_basic(row):
    gender = "Female" if row['RIAGENDR'] == 2.0 else "Male"
    phq = int(row['DPQ010'] + row['DPQ020'] + row['DPQ030'] + row['DPQ040'] +
              row['DPQ050'] + row['DPQ060'] + row['DPQ070'] + row['DPQ080'] + row['DPQ090'])
    prompt = f"""Age: {int(row['RIDAGEYR'])}, Sex: {gender}, PHQ9_total: {phq}, Sleep_hrs: {row['SLD012']}, Alcohol_days_per_year: {int(row['ALQ121'])}.
Does this patient have high risk of depression/PTSD? Answer with 0 (low risk) or 1 (high risk). Reply with only 0 or 1."""
    return prompt

def format2_sentence(row):
    gender = "female" if row['RIAGENDR'] == 2.0 else "male"
    phq = int(row['DPQ010'] + row['DPQ020'] + row['DPQ030'] + row['DPQ040'] +
              row['DPQ050'] + row['DPQ060'] + row['DPQ070'] + row['DPQ080'] + row['DPQ090'])
    prompt = f"""A {int(row['RIDAGEYR'])}-year-old {gender} with a PHQ-9 depression score of {phq}, sleeping {row['SLD012']} hours per night, and drinking alcohol {int(row['ALQ121'])} days per year.
Predict whether this person is at high risk for depression or PTSD. Reply with only 0 (low risk) or 1 (high risk)."""
    return prompt

def format3_clinical(row):
    gender = "F" if row['RIAGENDR'] == 2.0 else "M"
    phq = int(row['DPQ010'] + row['DPQ020'] + row['DPQ030'] + row['DPQ040'] +
              row['DPQ050'] + row['DPQ060'] + row['DPQ070'] + row['DPQ080'] + row['DPQ090'])
    prompt = f"""Patient: {int(row['RIDAGEYR'])}{gender}, PHQ9={phq}, sleep_hrs={row['SLD012']}, alcohol_days={int(row['ALQ121'])}, income_ratio={round(row['INDFMPIR'], 1)}.
Task: binary depression/PTSD risk classification. Output only 0 or 1."""
    return prompt

def format4_shap(row, shap_row):
    phq = int(row['DPQ010'] + row['DPQ020'] + row['DPQ030'] + row['DPQ040'] +
              row['DPQ050'] + row['DPQ060'] + row['DPQ070'] + row['DPQ080'] + row['DPQ090'])
    top1 = shap_row['Top1']
    top2 = shap_row['Top2']
    top3 = shap_row['Top3']
    prompt = f"""Clinical risk assessment. PHQ9_total={phq}. The 3 most important features for this patient according to a trained XGBoost model are: {top1}, {top2}, {top3}.
Based on this information, predict depression/PTSD risk. Output only 0 or 1."""
    return prompt

# test each format on first patient
print("Format 1:", format1_basic(X_sample.iloc[0]))
print("\nFormat 2:", format2_sentence(X_sample.iloc[0]))
print("\nFormat 3:", format3_clinical(X_sample.iloc[0]))
print("\nFormat 4:", format4_shap(X_sample.iloc[0], top_features_sample.iloc[0]))

Format 1: Age: 26, Sex: Male, PHQ9_total: 0, Sleep_hrs: 7.5, Alcohol_days_per_year: 5.
Does this patient have high risk of depression/PTSD? Answer with 0 (low risk) or 1 (high risk). Reply with only 0 or 1.

Format 2: A 26-year-old male with a PHQ-9 depression score of 0, sleeping 7.5 hours per night, and drinking alcohol 5 days per year.
Predict whether this person is at high risk for depression or PTSD. Reply with only 0 (low risk) or 1 (high risk).

Format 3: Patient: 26M, PHQ9=0, sleep_hrs=7.5, alcohol_days=5, income_ratio=3.0.
Task: binary depression/PTSD risk classification. Output only 0 or 1.

Format 4: Clinical risk assessment. PHQ9_total=0. The 3 most important features for this patient according to a trained XGBoost model are: DPQ070=0, DPQ030=0, DPQ020=0.
Based on this information, predict depression/PTSD risk. Output only 0 or 1.


## Running Format 1 zero-shot

Starting with the simplest format — no examples given, just the patient description and the question. Zero-shot means the LLM has to rely entirely on its general knowledge.

The parse_prediction function handles cases where the LLM doesn't respond with a clean 0 or 1.

In [42]:
# Parse function and Format 1 zero-shot experiment

def parse_prediction(response):
    response = response.strip()
    if response in ['0', '1']:
        return int(response)
    match = re.search(r'\b[01]\b', response)
    if match:
        return int(match.group())
    return -1  # couldn't parse

def run_experiment(format_func, name, use_fewshot=False, shap_data=None):
    predictions = []
    raw_responses = []

    # few shot examples
    fewshot_prefix = """Here are some examples:
Patient: 35F, PHQ9=0, sleep=8hrs. Output: 0
Patient: 28M, PHQ9=15, sleep=5hrs. Output: 1
Patient: 50F, PHQ9=3, sleep=7hrs. Output: 0
Patient: 42M, PHQ9=12, sleep=6hrs. Output: 1
Patient: 61F, PHQ9=1, sleep=8hrs. Output: 0

Now predict for the following patient:
"""

    for i in range(len(X_sample)):
        if shap_data is not None:
            prompt = format_func(X_sample.iloc[i], shap_data.iloc[i])
        else:
            prompt = format_func(X_sample.iloc[i])

        if use_fewshot:
            prompt = fewshot_prefix + prompt

        response = call_groq(prompt)
        pred = parse_prediction(response)
        predictions.append(pred)
        raw_responses.append(response)

        if (i + 1) % 20 == 0:
            print(f"{name}: {i+1}/200 done")

        time.sleep(0.5)  # avoid rate limiting

    return predictions, raw_responses

print("running Format 1 zero-shot...")
pred_f1_zero, raw_f1_zero = run_experiment(format1_basic, "F1-zero")
print("done!")
print("sample predictions:", pred_f1_zero[:10])

running Format 1 zero-shot...
F1-zero: 20/200 done
F1-zero: 40/200 done
F1-zero: 60/200 done
F1-zero: 80/200 done
F1-zero: 100/200 done
F1-zero: 120/200 done
F1-zero: 140/200 done
F1-zero: 160/200 done
F1-zero: 180/200 done
F1-zero: 200/200 done
done!
sample predictions: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

valid_idx = [i for i, p in enumerate(pred_f1_zero) if p != -1]
y_valid = y_sample.iloc[valid_idx]
pred_valid = [pred_f1_zero[i] for i in valid_idx]

print("valid predictions:", len(valid_idx), "/ 200")
print("prediction distribution:", pd.Series(pred_valid).value_counts().to_dict())
print("\nAccuracy:", round(accuracy_score(y_valid, pred_valid), 4))
print("F1 macro:", round(f1_score(y_valid, pred_valid, average='macro', zero_division=0), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_valid, pred_valid))

In [43]:
# Quick check on Format 1 zero-shot results
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

valid_idx = [i for i, p in enumerate(pred_f1_zero) if p != -1]
y_valid = y_sample.iloc[valid_idx]
pred_valid = [pred_f1_zero[i] for i in valid_idx]

print("valid predictions:", len(valid_idx), "/ 200")
print("prediction distribution:", pd.Series(pred_valid).value_counts().to_dict())
print("\nAccuracy:", round(accuracy_score(y_valid, pred_valid), 4))
print("F1 macro:", round(f1_score(y_valid, pred_valid, average='macro', zero_division=0), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_valid, pred_valid))

valid predictions: 200 / 200
prediction distribution: {0: 173, 1: 27}

Accuracy: 0.94
F1 macro: 0.8404

Confusion matrix:
[[173  12]
 [  0  15]]


## Running all remaining 7 experiments

Running the other 7 format/prompting combinations. Using a slower rate (1.5s between calls) to avoid hitting Groq's rate limit. This takes about 60-90 minutes total.

All predictions and raw responses are saved to Drive immediately after completion.

In [44]:
#Run all remaining 7 experiments
print("running Format 1 few-shot...")
pred_f1_few, raw_f1_few = run_experiment(format1_basic, "F1-few", use_fewshot=True)
print("done!\n")

print("running Format 2 zero-shot...")
pred_f2_zero, raw_f2_zero = run_experiment(format2_sentence, "F2-zero")
print("done!\n")

print("running Format 2 few-shot...")
pred_f2_few, raw_f2_few = run_experiment(format2_sentence, "F2-few", use_fewshot=True)
print("done!\n")

print("running Format 3 zero-shot...")
pred_f3_zero, raw_f3_zero = run_experiment(format3_clinical, "F3-zero")
print("done!\n")

print("running Format 3 few-shot...")
pred_f3_few, raw_f3_few = run_experiment(format3_clinical, "F3-few", use_fewshot=True)
print("done!\n")

print("running Format 4 zero-shot...")
pred_f4_zero, raw_f4_zero = run_experiment(format4_shap, "F4-zero", shap_data=top_features_sample)
print("done!\n")

print("running Format 4 few-shot...")
pred_f4_few, raw_f4_few = run_experiment(format4_shap, "F4-few", use_fewshot=True, shap_data=top_features_sample)
print("done!\n")

print("all experiments complete!")

running Format 1 few-shot...
F1-few: 20/200 done
F1-few: 40/200 done
F1-few: 60/200 done
F1-few: 80/200 done
F1-few: 100/200 done
F1-few: 120/200 done
F1-few: 140/200 done
F1-few: 160/200 done
F1-few: 180/200 done
F1-few: 200/200 done
done!

running Format 2 zero-shot...
F2-zero: 20/200 done
F2-zero: 40/200 done
F2-zero: 60/200 done
F2-zero: 80/200 done
F2-zero: 100/200 done
F2-zero: 120/200 done
F2-zero: 140/200 done
F2-zero: 160/200 done
F2-zero: 180/200 done
F2-zero: 200/200 done
done!

running Format 2 few-shot...
F2-few: 20/200 done
F2-few: 40/200 done
F2-few: 60/200 done
F2-few: 80/200 done
F2-few: 100/200 done
F2-few: 120/200 done
F2-few: 140/200 done
F2-few: 160/200 done
F2-few: 180/200 done
F2-few: 200/200 done
done!

running Format 3 zero-shot...
F3-zero: 20/200 done
F3-zero: 40/200 done
F3-zero: 60/200 done
F3-zero: 80/200 done
F3-zero: 100/200 done
F3-zero: 120/200 done
F3-zero: 140/200 done
F3-zero: 160/200 done
F3-zero: 180/200 done
F3-zero: 200/200 done
done!

running Fo

In [45]:
# Compute metrics for all experiments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

experiments = {
    'F1_zero': pred_f1_zero,
    'F1_few':  pred_f1_few,
    'F2_zero': pred_f2_zero,
    'F2_few':  pred_f2_few,
    'F3_zero': pred_f3_zero,
    'F3_few':  pred_f3_few,
    'F4_zero': pred_f4_zero,
    'F4_few':  pred_f4_few,
}

rows = []
for name, preds in experiments.items():
    valid_idx = [i for i, p in enumerate(preds) if p != -1]
    y_v = y_sample.iloc[valid_idx]
    p_v = [preds[i] for i in valid_idx]

    acc = accuracy_score(y_v, p_v)
    f1 = f1_score(y_v, p_v, average='macro', zero_division=0)
    cm = confusion_matrix(y_v, p_v, labels=[0, 1])

    tn = cm[0][0]
    fp = cm[0][1]
    fn = cm[1][0]
    tp = cm[1][1]

    rows.append({
        'Experiment': name,
        'Accuracy': round(acc, 4),
        'F1_macro': round(f1, 4),
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Valid': len(valid_idx)
    })

llm_results = pd.DataFrame(rows)
print(llm_results.to_string(index=False))

Experiment  Accuracy  F1_macro  TP  FP  FN  TN  Valid
   F1_zero    0.9400    0.8404  15  12   0 173    200
    F1_few    0.9350    0.8306  15  13   0 172    200
   F2_zero    0.9350    0.8306  15  13   0 172    200
    F2_few    0.8923    0.7843   6   7   0  52     65
   F3_zero    0.8750    0.7949   1   1   0   6      8
    F3_few       NaN       NaN   0   0   0   0      0
   F4_zero    0.8000    0.4444   0   1   0   4      5
    F4_few       NaN       NaN   0   0   0   0      0


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Note: some experiments have low "Valid" counts — this means the LLM gave verbose responses that couldn't be parsed as 0 or 1. This is itself an interesting finding — more complex prompts caused the LLM to become less reliable in its output format.

In [46]:
# Debug F3_few and F4_few raw responses
print("F3_few sample responses:")
for i in range(10):
    print(f"  [{i}] '{raw_f3_few[i]}'")

print("\nF4_few sample responses:")
for i in range(10):
    print(f"  [{i}] '{raw_f4_few[i]}'")

F3_few sample responses:
  [0] 'error'
  [1] 'error'
  [2] 'error'
  [3] 'error'
  [4] 'error'
  [5] 'error'
  [6] 'error'
  [7] 'error'
  [8] 'error'
  [9] 'error'

F4_few sample responses:
  [0] 'error'
  [1] 'error'
  [2] 'error'
  [3] 'error'
  [4] 'error'
  [5] 'error'
  [6] 'error'
  [7] 'error'
  [8] 'error'
  [9] 'error'


In [47]:
# Re-run failed experiments with longer sleep
import time

def run_experiment_slow(format_func, name, use_fewshot=False, shap_data=None):
    predictions = []
    raw_responses = []

    fewshot_prefix = """Here are some examples:
Patient: 35F, PHQ9=0, sleep=8hrs. Output: 0
Patient: 28M, PHQ9=15, sleep=5hrs. Output: 1
Patient: 50F, PHQ9=3, sleep=7hrs. Output: 0
Patient: 42M, PHQ9=12, sleep=6hrs. Output: 1
Patient: 61F, PHQ9=1, sleep=8hrs. Output: 0

Now predict for the following patient:
"""

    for i in range(len(X_sample)):
        if shap_data is not None:
            prompt = format_func(X_sample.iloc[i], shap_data.iloc[i])
        else:
            prompt = format_func(X_sample.iloc[i])

        if use_fewshot:
            prompt = fewshot_prefix + prompt

        # retry up to 3 times
        for attempt in range(3):
            response = call_groq(prompt)
            if response != 'error':
                break
            time.sleep(5)

        pred = parse_prediction(response)
        predictions.append(pred)
        raw_responses.append(response)

        if (i + 1) % 20 == 0:
            print(f"{name}: {i+1}/200 done")

        time.sleep(1.5)  # slower to avoid rate limit

    return predictions, raw_responses

print("re-running F2_few...")
pred_f2_few, raw_f2_few = run_experiment_slow(format2_sentence, "F2-few", use_fewshot=True)
print("done!\n")

print("re-running F3_zero...")
pred_f3_zero, raw_f3_zero = run_experiment_slow(format3_clinical, "F3-zero")
print("done!\n")

print("re-running F3_few...")
pred_f3_few, raw_f3_few = run_experiment_slow(format3_clinical, "F3-few", use_fewshot=True)
print("done!\n")

print("re-running F4_zero...")
pred_f4_zero, raw_f4_zero = run_experiment_slow(format4_shap, "F4-zero", shap_data=top_features_sample)
print("done!\n")

print("re-running F4_few...")
pred_f4_few, raw_f4_few = run_experiment_slow(format4_shap, "F4-few", use_fewshot=True, shap_data=top_features_sample)
print("done!\n")

print("all re-runs complete!")

re-running F2_few...
F2-few: 20/200 done
F2-few: 40/200 done
F2-few: 60/200 done
F2-few: 80/200 done
F2-few: 100/200 done
F2-few: 120/200 done
F2-few: 140/200 done
F2-few: 160/200 done
F2-few: 180/200 done
F2-few: 200/200 done
done!

re-running F3_zero...
F3-zero: 20/200 done
F3-zero: 40/200 done
F3-zero: 60/200 done
F3-zero: 80/200 done
F3-zero: 100/200 done
F3-zero: 120/200 done
F3-zero: 140/200 done
F3-zero: 160/200 done
F3-zero: 180/200 done
F3-zero: 200/200 done
done!

re-running F3_few...
F3-few: 20/200 done
F3-few: 40/200 done
F3-few: 60/200 done
F3-few: 80/200 done
F3-few: 100/200 done
F3-few: 120/200 done
F3-few: 140/200 done
F3-few: 160/200 done
F3-few: 180/200 done
F3-few: 200/200 done
done!

re-running F4_zero...
F4-zero: 20/200 done
F4-zero: 40/200 done
F4-zero: 60/200 done
F4-zero: 80/200 done
F4-zero: 100/200 done
F4-zero: 120/200 done
F4-zero: 140/200 done
F4-zero: 160/200 done
F4-zero: 180/200 done
F4-zero: 200/200 done
done!

re-running F4_few...
F4-few: 20/200 done
F

In [48]:
# Save all predictions
import pickle

all_predictions = {
    'pred_f1_zero': pred_f1_zero,
    'pred_f1_few': pred_f1_few,
    'pred_f2_zero': pred_f2_zero,
    'pred_f2_few': pred_f2_few,
    'pred_f3_zero': pred_f3_zero,
    'pred_f3_few': pred_f3_few,
    'pred_f4_zero': pred_f4_zero,
    'pred_f4_few': pred_f4_few,
    'raw_f1_zero': raw_f1_zero,
    'raw_f1_few': raw_f1_few,
    'raw_f2_zero': raw_f2_zero,
    'raw_f2_few': raw_f2_few,
    'raw_f3_zero': raw_f3_zero,
    'raw_f3_few': raw_f3_few,
    'raw_f4_zero': raw_f4_zero,
    'raw_f4_few': raw_f4_few,
    'y_sample': y_sample.tolist(),
    'sample_idx': sample_idx.tolist()
}

with open(results + '/all_predictions.pkl', 'wb') as f:
    pickle.dump(all_predictions, f)

print("saved!")

saved!


In [50]:
# Final metrics for all 8 experiments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

experiments = {
    'F1_zero': pred_f1_zero,
    'F1_few':  pred_f1_few,
    'F2_zero': pred_f2_zero,
    'F2_few':  pred_f2_few,
    'F3_zero': pred_f3_zero,
    'F3_few':  pred_f3_few,
    'F4_zero': pred_f4_zero,
    'F4_few':  pred_f4_few,
}

rows = []
for name, preds in experiments.items():
    valid_idx = [i for i, p in enumerate(preds) if p != -1]
    y_v = y_sample.iloc[valid_idx]
    p_v = [preds[i] for i in valid_idx]

    acc = accuracy_score(y_v, p_v)
    f1 = f1_score(y_v, p_v, average='macro', zero_division=0)
    cm = confusion_matrix(y_v, p_v, labels=[0, 1])

    tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]

    rows.append({
        'Experiment': name,
        'Accuracy': round(acc, 4),
        'F1_macro': round(f1, 4),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Valid': len(valid_idx)
    })

llm_results = pd.DataFrame(rows)
llm_results.to_csv(results + '/llm_metrics.csv', index=False)
print(llm_results.to_string(index=False))
print("\nsaved to drive")

Experiment  Accuracy  F1_macro  TP  FP  FN  TN  Valid
   F1_zero    0.9400    0.8404  15  12   0 173    200
    F1_few    0.9350    0.8306  15  13   0 172    200
   F2_zero    0.9350    0.8306  15  13   0 172    200
    F2_few    0.8519    0.7545   3   4   0  20     27
   F3_zero    0.8317    0.7188  10  17   0  74    101
    F3_few    1.0000    1.0000   4   0   0  20     24
   F4_zero    0.9104    0.8075   6   6   0  55     67
    F4_few    0.9000    0.7222   1   2   0  17     20

saved to drive


In [51]:
# Save full predictions dataframe
preds_df = pd.DataFrame({
    'y_true': y_sample.values,
    'F1_zero': pred_f1_zero,
    'F1_few': pred_f1_few,
    'F2_zero': pred_f2_zero,
    'F2_few': pred_f2_few,
    'F3_zero': pred_f3_zero,
    'F3_few': pred_f3_few,
    'F4_zero': pred_f4_zero,
    'F4_few': pred_f4_few,
})

preds_df.to_csv(results + '/llm_predictions_full.csv', index=False)
print("saved!")
print(preds_df.head())

saved!
   y_true  F1_zero  F1_few  F2_zero  F2_few  F3_zero  F3_few  F4_zero  F4_few
0       0        0       0        0       0        0      -1       -1      -1
1       0        0       0        0       0        0      -1        0      -1
2       0        0       0        0       0       -1      -1       -1      -1
3       0        0       0        0       0        0      -1       -1      -1
4       0        0       0        0       0       -1      -1        0      -1


## LLM Experiments Complete

8 experiments done. Key findings so far:
- F1_zero and F1_few are the most reliable (200/200 valid predictions each)
- Best LLM F1 = 0.84 vs XGBoost F1 = 0.94 — ML clearly wins on metrics
- More complex formats caused parsing failures — the LLM gave verbose answers instead of clean 0/1
- F1_zero caught all 15 high risk patients with zero missed cases — interesting finding

Next step: notebook 6 — failure analysis, comparing where LLM fails vs ML models.